In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
import pandas as pd
import os
food_delivery_path = os.path.join(path, 'Q1_data.csv')
df_foods = pd.read_csv(food_delivery_path)

In [ ]:
# Task 2: Write your code here:
df_foods.head()

In [ ]:
# Task 3: Write your code here:
df_foods.info()

In [ ]:
# Task 4: Write your code here:
df_foods.describe()

In [ ]:
# Task 5: Write your code here:
import matplotlib.pyplot as plt
df_foods['Delivery_Time'].hist(bins=20)
plt.title('Delivery Time Distribution')
plt.show()

In [ ]:
# Task 1: Write your code here:
df_foods.drop(columns=['Order_ID'])

In [ ]:
# Task 2: Write your code here:
df_clean = df_foods.isnull().sum()
print(df_clean)

In [ ]:
df_clean = df_clean.dropna(subset=['Weather', 'Traffic_Level','Time_of_Day'])
df_clean['Delivery_Time'] = df_clean['Delivery_Time'].fillna(df_clean['Delivery_Time'].mean())
df_clean['Courier_Experience_yrs'] = df_clean['Courier_Experience_yrs'].fillna(df_clean['Courier_Experience_yrs'].mode()[0])


print("Missing values remaining:", df_clean.isnull().sum().sum())

In [ ]:
# Task 3: Write your code here:
print("\n--- Duplicates ---")
dupes = df_clean.duplicated().sum()
print(f"Number of duplicates: {dupes}")

In [ ]:
if dupes > 0:
    df_clean.drop_duplicates(inplace=True)
    print("Duplicates dropped.")

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import StandardScaler, LabelEncoder
categorical_cols = ['Weather', 'Traffic_Level', 'Time_of_Day', 'Vehicle_Type']
for col in categorical_cols:
    le = LabelEncoder()
    df_clean[col] = le.fit_transform(df_clean[col].astype(str))

df_clean.head()

In [ ]:
# Task 5: Write your code here:
from sklearn.model_selection import train_test_split, KFold
#X = df_clean.drop('Delivery_Time', axis=1)
#y = df_clean['Delivery_Time']

#X_train, X_test, y_train, y_test = train_test_split(
 #   X, y, test_size=0.2, random_state=42, shuffle=True
#)

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)
print(f"\nScaled ranges - Min: {X_train_scaled.min():.2f}, Max: {X_train_scaled.max():.2f}")
pd.DataFrame(X_train_scaled, columns=X_train.columns).head(3)

In [ ]:
# Task 6: Write your code here:
print(df_foods['Delivery_Time'].value_counts())

In [ ]:
#imbalance

In [ ]:
# Task 1: Write your code here:
# Define features (X) and target (y)
feature_cols = ['Distance_km',	'Weather',	'Traffic_Level',	'Time_of_Day	Vehicle_Type',	'Preparation_Time_min	,Courier_Experience_yrs']
X = df_clean[feature_cols]
y = df_clean['Delivery_Time']


In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestRegressor

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)
model.fit(X_train_scaled, y_train)
print("Model trained!")

y_pred = model.predict(X_test_scaled)


mae = mean_absolute_error(y_test, y_pred)


print(f"MAE:  ${mae:,.2f}")


In [ ]:
kfold = KFold(n_splits=5, shuffle=True, random_state=42, stratify=y)

mae_scores = []

for train_idx, val_idx in kfold.split(X_train_scaled):
    X_fold_train, X_fold_val = X_train_scaled[train_idx], X_train_scaled[val_idx]
    y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    # Train and predict
    model.fit(X_fold_train, y_fold_train)
    y_fold_pred = model.predict(X_fold_val)

    # Calculate metrics
    mae_scores.append(mean_absolute_error(y_fold_val, y_fold_pred))
    rmse_scores.append(np.sqrt(mean_squared_error(y_fold_val, y_fold_pred)))

mae_scores = np.array(mae_scores)
rmse_scores = np.array(rmse_scores)

print(f"5-Fold CV Results:")
print(f"MAE:  ${mae_scores.mean():,.2f}")
print(f"RMSE: ${rmse_scores.mean():,.2f}")

In [ ]:
# Task 1: Write your code here:
# Feature importance
importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(importance['feature'], importance['importance'], color='purple')
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:
y_pred = model.predict(X_test_scaled) # الحل

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(classification_report(y_test, y_pred, target_names=['Normal', 'Legendary']))


In [ ]:
# Task Bonus: Write your code here: